In [1]:
%pip install -q azure-ai-ml azure-identity


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 1: setup - run once
%pip install -q huggingface_hub azureml-fsspec adlfs azure-identity azure-ai-ml azure-storage-blob datasets

from huggingface_hub import HfApi, login
# paste your HF write token - get from https://huggingface.co/settings/tokens
# need WRITE access for esthxy/rlvr_group_correlation
login(token="hf_xxxxxxxxxxx") 

api = HfApi()
print("Logged in as:", api.whoami()["name"])

In [1]:
import os

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)
print(os.getcwd())


/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/rlvr_group_correlation


In [2]:
!python submit_stage1.py --step env

Traceback (most recent call last):
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/rlvr_group_correlation/submit_stage1.py", line 15, in <module>
    from azure.ai.ml import MLClient, load_job
ModuleNotFoundError: No module named 'azure.ai.ml'


In [5]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Initialize the ML Client with subscription ID, resource group, and workspace name
ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)

# Loop through environments to find and print the latest version details
for n in ["rlvr-stage1-cpu", "rlvr-stage1-gpu"]:
    v = max(int(e.version) for e in ml_client.environments.list(name=n))
    e = ml_client.environments.get(n, version=str(v))
    print(n, "v" + str(v), "|", e.creation_context.created_at)


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


ResourceNotFoundError: (UserError) System.Net.Http.HttpConnectionResponseContent
Code: UserError
Message: System.Net.Http.HttpConnectionResponseContent

In [9]:
import sys,os

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)
print(sys.executable)

/anaconda/envs/azureml_py310_sdkv2/bin/python


In [10]:
!{sys.executable} submit_stage1.py --step env

[env] created/updated rlvr-stage1-cpu
[env] created/updated rlvr-stage1-gpu


In [11]:
import glob
import json
import os
import sys
import time

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def watch(job_name, poll=60):
    """Poll instead of stream — survives notebook disconnects."""
    while True:
        s = ml_client.jobs.get(job_name).status
        print(time.strftime("%H:%M:%S"), job_name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [12]:
for n in ["rlvr-stage1-cpu", "rlvr-stage1-gpu"]:
    vs = [int(e.version) for e in ml_client.environments.list(name=n)]
    print(n, "latest v" + str(max(vs)))


rlvr-stage1-cpu latest v1
rlvr-stage1-gpu latest v1


In [13]:
!{sys.executable} submit_stage1.py --step compute

[compute] cpu-audit-cluster: Standard_D4ds_v5 state=Succeeded min=0 max=5
[compute] t4v3: Standard_NC4as_T4_v3 state=Succeeded min=0 max=1
[compute] ralir-v3d-control: Standard_E48s_v3 state=Succeeded min=- max=-


In [14]:
c = ml_client.compute.get("t4v3")
c.min_instances = 0
c.idle_time_before_scale_down = 900

ml_client.compute.begin_create_or_update(c).result()

print("t4v3 min_instances =", ml_client.compute.get("t4v3").min_instances)


KeyboardInterrupt: 

In [15]:
!{sys.executable} submit_stage1.py --step prompts

Traceback (most recent call last):
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/azure/ai/ml/_utils/utils.py", line 313, in load_yaml
    cm = open(source, "r", encoding=DefaultOpenEncoding.READ)
FileNotFoundError: [Errno 2] No such file or directory: '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/rlvr_group_correlation/jobs/01_build_prompts.yml'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/rlvr_group_correlation/submit_stage1.py", line 89, in <module>
    run("01_build_prompts.yml")
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/code/Users/estherxin0011/rlvr_group_correlation/submit_stage1.py", line 69, in run
    job = load_job(os.path.join(PROJ, "jobs", yml))
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages

In [16]:
JOB_PROMPTS = "PASTE_JOB_NAME_FROM_ABOVE"
watch(JOB_PROMPTS)

ResourceNotFoundError: (UserError) Job PASTE_JOB_NAME_FROM_ABOVE not found.
Code: UserError
Message: Job PASTE_JOB_NAME_FROM_ABOVE not found.

In [17]:
JOB_PROMPTS = submit("01_build_prompts.yml")
watch(JOB_PROMPTS)
logs(JOB_PROMPTS, "[prompts]")

NameError: name 'submit' is not defined

In [1]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)


helpers ready: <function submit at 0x7c4019f476d0> <function watch at 0x7c4019f47520> <function logs at 0x7c4019f47760>


In [20]:
JOB_PROMPTS = submit("01 build prompts.yml")
watch(JOB_PROMPTS)
logs(JOB_PROMPTS, "[prompts]")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.03 MBs): 100%|███████

submitted: coral_stamp_q2whsfzrxl 
 https://ml.azure.com/runs/coral_stamp_q2whsfzrxl?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
09:14:12 coral_stamp_q2whsfzrxl Starting
09:15:12 coral_stamp_q2whsfzrxl Queued
09:16:13 coral_stamp_q2whsfzrxl Queued
09:17:13 coral_stamp_q2whsfzrxl Queued
09:18:14 coral_stamp_q2whsfzrxl Running
09:19:14 coral_stamp_q2whsfzrxl Running
09:20:15 coral_stamp_q2whsfzrxl Finalizing
09:21:17 coral_stamp_q2whsfzrxl Completed
[prompts] wrote 49864 -> /mnt/azureml/cr/j/81962bb625c248a9b779c5bf95cd8eef/cap/data-capability/wd/prompt_pool/prompts_50k.jsonl
[prompts] by source: Counter({'deepmath': 34895, 'math': 7496, 'gsm8k': 7473})


In [4]:
JOB_PILOT = submit("02 pilot generate.yml")
watch(JOB_PILOT)
logs(JOB_PILOT, "[gen]")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

submitted: careful_rabbit_5jzv24kwy2 
 https://ml.azure.com/runs/careful_rabbit_5jzv24kwy2?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
10:46:40 careful_rabbit_5jzv24kwy2 Starting
10:47:40 careful_rabbit_5jzv24kwy2 Queued
10:48:41 careful_rabbit_5jzv24kwy2 Queued
10:49:41 careful_rabbit_5jzv24kwy2 Queued
10:50:42 careful_rabbit_5jzv24kwy2 Queued
10:51:44 careful_rabbit_5jzv24kwy2 Queued
10:52:44 careful_rabbit_5jzv24kwy2 Queued
10:53:45 careful_rabbit_5jzv24kwy2 Queued
10:54:45 careful_rabbit_5jzv24kwy2 Queued
10:55:46 careful_rabbit_5jzv24kwy2 Queued
10:56:47 careful_rabbit_5jzv24kwy2 Queued
10:57:47 careful_rabbit_5jzv24kwy2 Queued
10:58:47 careful_rabbit_5jzv24kwy2 Queued
10:59:48 careful_rabbit_5jzv24kwy2 Queued
11:00:48 careful_rabbit_5jzv24kwy2 Queued
11:01:49 careful_rabbit_5jzv24kwy2 Queued


KeyboardInterrupt: 

In [5]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=120):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for l in open(f, errors="ignore"):
            if needle in l:
                print(l.rstrip())


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [6]:
ml_client.environments.create_or_update(
    Environment(
        name="rlvr-stage1-cpu-torch",
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
        conda_file=os.path.join(PROJ, "env/stage1-cpu-torch.yml"),
    )
)
print("registered")


registered


In [10]:
JOB_PILOT = submit("02b_pilot_generate_cpu.yml")
watch(JOB_PILOT, poll=180)
logs(JOB_PILOT, "[cpu-gen]")

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: sweet_rabbit_r8dkr7zxhw 
 https://ml.azure.com/runs/sweet_rabbit_r8dkr7zxhw?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
12:55:38 sweet_rabbit_r8dkr7zxhw Starting
12:58:38 sweet_rabbit_r8dkr7zxhw Running
13:01:40 sweet_rabbit_r8dkr7zxhw Running
13:04:41 sweet_rabbit_r8dkr7zxhw Running
13:07:42 sweet_rabbit_r8dkr7zxhw Running
13:10:43 sweet_rabbit_r8dkr7zxhw Running
13:13:44 sweet_rabbit_r8dkr7zxhw Running


In [ ]:
from azure.ai.ml.entities import Data
ml_client.data.create_or_update(Data(
name="tmp-pilot-list", version="1", type="uri_folder",
path="azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/rollouts_pilot/"))

In [14]:
JOB_GATE = submit("03 pilot gate.yml")
watch(JOB_GATE)

Uploading src (0.04 MBs): 100%|██████████| 35031/35031 [00:00<00:00, 648511.53it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: frosty_board_v7vlh8vqbn 
 https://ml.azure.com/runs/frosty_board_v7vlh8vqbn?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
15:59:50 frosty_board_v7vlh8vqbn Starting
16:01:51 frosty_board_v7vlh8vqbn Failed


'Failed'

In [2]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


# --- 1. verify the new pilot_gate.py parses locally (catches paste errors) ---
ast.parse(open(f"{PROJ}/src/pilot_gate.py").read())
print("pilot_gate.py syntax OK")

# --- 2. write the gate job YAML (scoped to the 0.5B pilot only) -------------
gate_yaml = """$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
type: command
display_name: stage1-03c-PILOT-GATE
experiment_name: rlvr-group-correlation

code: ../src
command: >-
  python pilot_gate.py
  --rollouts ${{inputs.rollouts}}
  --out ${{outputs.gate}}/gate_report.json
  --only_model qwen2.5-0.5b-cpu-pilot
  --min_mixed 0.30
  --max_cat_share 0.98
  --min_disagree 0.02

environment: azureml:rlvr-stage1-cpu@latest
compute: azureml:cpu-audit-cluster

inputs:
  rollouts:
    type: uri_folder
    mode: ro_mount
    path: azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/rollouts_pilot/

outputs:
  gate:
    type: uri_folder
    mode: rw_mount
    path: azureml://datastores/workspaceblobstore/paths/rlvr_group_corr/gate/
"""
open(f"{PROJ}/jobs/03c_pilot_gate.yml", "w").write(gate_yaml)
print("wrote jobs/03c_pilot_gate.yml")

# --- 3. submit and wait ----------------------------------------------------
JOB_GATE = submit("03c_pilot_gate.yml")
status = watch(JOB_GATE)
print("\njob status:", status, " (Failed may mean a GATE failed, exit 2)")

# --- 4. print the log so gate lines are visible either way ------------------
d = f"./_logs_{JOB_GATE}"
ml_client.jobs.download(JOB_GATE, download_path=d, all=True)
for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
    for line in open(f, errors="ignore"):
        if any(
            t in line
            for t in (
                "[gate]",
                "GATE ",
                "DO NOT PROCEED",
                "All gates clear",
                "Error",
                "Traceback",
            )
        ):
            print(line.rstrip())

# --- 5. read the report ----------------------------------------------------
try:
    ml_client.jobs.download(
        JOB_GATE, download_path="./_gate", output_name="gate"
    )
    r = json.load(
        open(glob.glob("./_gate/**/gate_report.json", recursive=True)[0])
    )

    print("\n" + "=" * 64)
    print(
        f"files {r['n_files']} | groups {r['n_groups']} | models {r['models_present']}"
    )
    print(
        f"G1 mixed-group rate {r['gate1_mixed_group_rate']:<8} >= {r['gate1_threshold']} "
        f"{'PASS' if r['gate1_pass'] else 'FAIL'}"
    )
    print(
        f"G2 mean cat share {r['gate2_mean_primary_cat_share']:<8} < {r['gate2_threshold_max_share']} "
        f"{'PASS' if r['gate2_pass'] else 'FAIL'}"
    )
    print(
        f"G3 disagreement rate {r['gate3_verifier_disagreement_rate']:<8} >= {r['gate3_threshold']} "
        f"{'PASS' if r['gate3_pass'] else 'FAIL'}"
    )
    print(f"\nALL GATES PASS: {r['ALL_GATES_PASS']}")
    print(
        f"prelim rho_cat {r['preliminary_rho_cat_proxy']} CI95 {r['preliminary_rho_ci95']}"
    )
    print(f"Kish n_eff / k {r['kish_n_eff_of_k']} of {r['k_bar']}")
    print("\nrho by primary category:")
    print(json.dumps(r["rho_by_primary_cat"], indent=2))
    print("\ncategory distribution:")
    print(json.dumps(r["primary_cat_distribution"], indent=2))
    print("=" * 64)
except Exception as e:
    print("\ncould not read gate_report.json:", type(e).__name__, e)


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

submitted: loyal_rabbit_lw1xl4b0dn 
 https://ml.azure.com/runs/loyal_rabbit_lw1xl4b0dn?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
18:31:25 loyal_rabbit_lw1xl4b0dn Starting


KeyboardInterrupt: 

In [18]:
print(ml_client.compute.get("t4v3").provisioning_state) # need Succeeded + free
JOB_04 = submit("04 full.yml")
print(JOB_04)

Succeeded


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


In [1]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)

JOB_GATE = submit("03d_gate_real.yml")
watch(JOB_GATE)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

submitted: loving_carpet_r9crwvd6rv 
 https://ml.azure.com/runs/loving_carpet_r9crwvd6rv?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
19:59:09 loving_carpet_r9crwvd6rv Starting
19:59:39 loving_carpet_r9crwvd6rv Queued
20:00:09 loving_carpet_r9crwvd6rv Queued


In [3]:
# ============================================================
# STAGE 1 — GATE (job 03) : full runnable cell
# Re-run this whole cell after any kernel restart.
# ============================================================
import ast
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=30):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)
ml_client.jobs.download(JOB_04, download_path="./_out_1p5b", output_name="rollouts")
import glob, json
for f in glob.glob("./_out_1p5b/**/_summary_qwen2.5-1.5b.json", recursive=True):print(json.load(open(f)))

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


NameError: name 'JOB_04' is not defined

In [4]:
JOB_05 = submit("05_full_generate_7b_awq.yml")
print(JOB_05)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: sharp_scooter_glw2sbq850 
 https://ml.azure.com/runs/sharp_scooter_glw2sbq850?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
sharp_scooter_glw2sbq850


In [6]:
JOB_04 = submit("04_full_generate_1p5b.yml")
ml_client.jobs.download(JOB_04, download_path="./_out_1p5b", output_name="rollouts")
import glob, json
for f in glob.glob("./_out_1p5b/**/_summary_qwen2.5-1.5b.json", recursive=True):print(json.load(open(f)))

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: careful_lime_nfzs56yjnq 
 https://ml.azure.com/runs/careful_lime_nfzs56yjnq?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42


JobException: This job is in state Starting. Download is allowed only in states ['Completed', 'Failed', 'Canceled', 'NotResponding', 'Paused']

In [1]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)


JOB_VERIFY = submit("06_rule_verify_1p5b.yml")
watch(JOB_VERIFY, poll=60)

helpers ready: <function submit at 0x70ba3a487880> <function watch at 0x70ba3a4876d0> <function logs at 0x70ba3a487910>
submitted: heroic_ice_pbvwtkyhdp 
 https://ml.azure.com/runs/heroic_ice_pbvwtkyhdp?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
21:52:19 heroic_ice_pbvwtkyhdp Starting
21:53:19 heroic_ice_pbvwtkyhdp Queued
21:54:20 heroic_ice_pbvwtkyhdp Queued
21:55:20 heroic_ice_pbvwtkyhdp Queued
21:56:21 heroic_ice_pbvwtkyhdp Running
21:57:22 heroic_ice_pbvwtkyhdp Finalizing
21:58:23 heroic_ice_pbvwtkyhdp Completed


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

'Completed'

In [2]:
JOB_REPLAY = submit("07_advantage_replay_1p5b.yml")
watch(JOB_REPLAY, poll=30)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: elated_cherry_c3vj1dbx99 
 https://ml.azure.com/runs/elated_cherry_c3vj1dbx99?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
21:59:09 elated_cherry_c3vj1dbx99 Starting
21:59:39 elated_cherry_c3vj1dbx99 Running
22:00:10 elated_cherry_c3vj1dbx99 Completed


'Completed'

In [3]:
ml_client.jobs.download(JOB_REPLAY, download_path="./_replay", output_name="report")
r = json.load(open(glob.glob("./_replay/**/replay_report_1p5b.json", recursive=True)[0]))
print(json.dumps(r, indent=2))

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "n_groups": 24998,
  "k": 8,
  "per_verifier": {
    "strict": {
      "global_mean_reward_p": 0.2467,
      "degenerate_rate_observed": 0.6695,
      "jensen_predicted_degenerate": 0.1037,
      "excess_degenerate_over_jensen": 0.5657
    },
    "loose": {
      "global_mean_reward_p": 0.2599,
      "degenerate_rate_observed": 0.6479,
      "jensen_predicted_degenerate": 0.09,
      "excess_degenerate_over_jensen": 0.5579
    },
    "numeric": {
      "global_mean_reward_p": 0.2622,
      "degenerate_rate_observed": 0.6481,
      "jensen_predicted_degenerate": 0.0878,
      "excess_degenerate_over_jensen": 0.5603
    },
    "flex": {
      "global_mean_reward_p": 0.2625,
      "degenerate_rate_observed": 0.6476,
      "jensen_predicted_degenerate": 0.0876,
      "excess_degenerate_over_jensen": 0.56
    }
  },
  "pairwise": {
    "strict_vs_loose": {
      "sign_flip_rate": 0.001,
      "group_corruption_rate": 0.0053
    },
    "strict_vs_numeric": {
      "sign_flip_rate": 0.001

In [2]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)

JOB_GOLD = submit("08_gold_sampling_1p5b.yml")
watch(JOB_GOLD, poll=30)

helpers ready: <function submit at 0x7b4a25bca170> <function watch at 0x7b4a26dbb910> <function logs at 0x7b4a25bcb2e0>
submitted: amusing_gold_mzt4l2vyyw 
 https://ml.azure.com/runs/amusing_gold_mzt4l2vyyw?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
22:40:34 amusing_gold_mzt4l2vyyw Starting
22:41:05 amusing_gold_mzt4l2vyyw Queued
22:41:36 amusing_gold_mzt4l2vyyw Running
22:42:07 amusing_gold_mzt4l2vyyw Running
22:42:37 amusing_gold_mzt4l2vyyw Finalizing
22:43:08 amusing_gold_mzt4l2vyyw Completed


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


'Completed'

In [5]:
# after job 05 has ~4-8 shards
JOB_GATE_7B = submit("03e_gate_7b.yml")
watch(JOB_GATE_7B)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: polite_bird_bnxxhryq00 
 https://ml.azure.com/runs/polite_bird_bnxxhryq00?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
22:04:39 polite_bird_bnxxhryq00 Starting
22:05:40 polite_bird_bnxxhryq00 Failed


'Failed'

In [1]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())


print("helpers ready:", submit, watch, logs)

JOB_SELECT = submit("09_select_compass_sample.yml")
watch(JOB_SELECT, poll=30)


helpers ready: <function submit at 0x7a04c41f40d0> <function watch at 0x7a04c41f6a70> <function logs at 0x7a04c41f6b90>
submitted: dynamic_sheep_2j35pz54kr 
 https://ml.azure.com/runs/dynamic_sheep_2j35pz54kr?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
01:25:24 dynamic_sheep_2j35pz54kr Starting
01:25:55 dynamic_sheep_2j35pz54kr Queued
01:26:25 dynamic_sheep_2j35pz54kr Queued
01:26:55 dynamic_sheep_2j35pz54kr Queued
01:27:26 dynamic_sheep_2j35pz54kr Queued
01:27:56 dynamic_sheep_2j35pz54kr Queued
01:28:26 dynamic_sheep_2j35pz54kr Queued
01:28:57 dynamic_sheep_2j35pz54kr Queued
01:29:27 dynamic_sheep_2j35pz54kr Queued
01:29:58 dynamic_sheep_2j35pz54kr Running
01:30:28 dynamic_sheep_2j35pz54kr Running
01:30:59 dynamic_sheep_2j35pz54kr Running
01:31:29 dynamic_sheep_2j35pz54kr Running
01:32:00 dynamic_sheep_2j35pz54kr Completed


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.08 MBs): 100%|███████

'Completed'

In [3]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())
JOB_COMPASS = submit("10_compass_verify.yml")
watch(JOB_COMPASS, poll=120)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: careful_comb_x5mh6mmjft 
 https://ml.azure.com/runs/careful_comb_x5mh6mmjft?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
01:38:47 careful_comb_x5mh6mmjft Starting
01:40:47 careful_comb_x5mh6mmjft Queued
01:42:48 careful_comb_x5mh6mmjft Queued
01:44:49 careful_comb_x5mh6mmjft Queued
01:46:50 careful_comb_x5mh6mmjft Canceled


'Canceled'

In [5]:
import glob
import json
import os
import time

from azure.ai.ml import MLClient, load_job
from azure.identity import DefaultAzureCredential

PROJ = (
    "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ralir-v3d-control/"
    "code/Users/estherxin0011/rlvr_group_correlation"
)
os.chdir(PROJ)

ml_client = MLClient(
    DefaultAzureCredential(),
    "7a8513b6-ada2-4ad1-aee2-687fa5663c82",
    "AIModels",
    "Reinforcementinfra",
)


def submit(yml):
    j = ml_client.jobs.create_or_update(
        load_job(os.path.join(PROJ, "jobs", yml))
    )
    print("submitted:", j.name, "\n", j.studio_url)
    return j.name


def watch(name, poll=60):
    while True:
        s = ml_client.jobs.get(name).status
        print(time.strftime("%H:%M:%S"), name, s, flush=True)
        if s in ("Completed", "Failed", "Canceled"):
            return s
        time.sleep(poll)


def logs(name, needle):
    d = f"./_logs_{name}"
    ml_client.jobs.download(name, download_path=d, all=True)
    for f in glob.glob(f"{d}/**/std_log*.txt", recursive=True):
        for line in open(f, errors="ignore"):
            if needle in line:
                print(line.rstrip())
# 3. once job 10 shows Completed, run final analysis
JOB_FINAL = submit("11_final_analysis.yml")
watch(JOB_FINAL, poll=30)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading src (0.09 MBs): 100%|██████████| 85613/85613 [00:00<00:00, 2723533.12it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


submitted: kind_board_tbw68plrbg 
 https://ml.azure.com/runs/kind_board_tbw68plrbg?wsid=/subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourcegroups/AIModels/workspaces/Reinforcementinfra&tid=4033a39c-0759-4168-a40c-a9bc796b4f42
02:07:13 kind_board_tbw68plrbg Starting
02:07:44 kind_board_tbw68plrbg Finalizing
02:08:14 kind_board_tbw68plrbg Completed


'Completed'

In [ ]:
# 1. gate on what's there now, before stopping
JOB_GATE_7B = submit("03e_gate_7b.yml")
watch(JOB_GATE_7B)